# aggregate_baseline — Rekap Hasil 5-Fold Student Baseline

Jalankan **setelah kelima fold baseline selesai dievaluasi** (boleh juga sebagian — fold yang belum ada cuma jadi warning):

1. **Add Input**: kelima output notebook **eval** k-fold baseline (yang berisi `student_baseline_overall_fold{N}_9class.csv`). Tidak perlu dikumpulkan manual — CSV dicari otomatis rekursif di `/kaggle/input`.
2. **Run All** (CPU cukup, hitungan detik). Pakai *Save & Run All* kalau mau `kfold_baseline_summary.csv` tersimpan di tab Output.
3. Hasil: `kfold_baseline_summary.csv` + tabel markdown di bawah cell (mAP@0.5 & mAP@0.5:0.95 siang/malam + penurunan relatif `(mAP50_siang − mAP50_malam) / mAP50_siang × 100`, per fold + mean ± SD) — siap tempel.

Catatan: penurunan relatif di sini dalam **persen** (sama dengan Tabel 6 paper dan `aggregate_baseline_vs_kd`), sedangkan `kfold-aggregate` milik KD mencetaknya sebagai pecahan. SD = SD sampel (n − 1). Untuk perbandingan baseline vs KD per fold + uji berpasangan, pakai `aggregate_baseline_vs_kd`.


In [1]:
# =============== KONFIGURASI ===============
# Folder yang di-scan untuk mencari student_baseline_overall_fold{N}_9class.csv.
# Di Kaggle: add kelima output notebook eval baseline sebagai Input, biarkan default.
SEARCH_DIRS = ["/kaggle/input", "/kaggle/working"]
OUT_PATH    = "/kaggle/working/kfold_baseline_summary.csv"
# ===========================================

import csv
import os
import statistics

K = 5
COLS = ["map50_day", "map50_night", "map5095_day", "map5095_night", "rel_drop"]
HEADERS = {"map50_day": "mAP@0.5 siang", "map50_night": "mAP@0.5 malam",
           "map5095_day": "mAP@0.5:0.95 siang", "map5095_night": "mAP@0.5:0.95 malam",
           "rel_drop": "penurunan relatif (%)"}
ND = {"map50_day": 4, "map50_night": 4, "map5095_day": 4, "map5095_night": 4, "rel_drop": 2}


def find_csv(fold):
    """Cari CSV overall baseline fold N di mana pun dalam SEARCH_DIRS (rekursif)."""
    target = f"student_baseline_overall_fold{fold}_9class.csv"
    hits = []
    for base in SEARCH_DIRS:
        if not os.path.isdir(base):
            continue
        for root, _dirs, files in os.walk(base):
            if target in files:
                hits.append(os.path.join(root, target))
    if len(hits) > 1:
        print(f"[WARNING] {target} ditemukan {len(hits)}x, memakai yang pertama: {hits[0]}")
    return hits[0] if hits else None


def read_fold(path):
    rows = {}
    with open(path) as f:
        for r in csv.DictReader(f):
            if r["split"] in rows:
                print(f"[WARNING] {path}: split '{r['split']}' muncul lebih dari sekali -- memakai baris terakhir")
            rows[r["split"]] = r
    assert "daytime" in rows and "night" in rows, f"{path}: butuh baris daytime & night"
    d, n = rows["daytime"], rows["night"]
    m50d, m50n = float(d["map50"]), float(n["map50"])
    return {
        "map50_day": m50d, "map50_night": m50n,
        "map5095_day": float(d["map50_95"]), "map5095_night": float(n["map50_95"]),
        "rel_drop": (m50d - m50n) / m50d * 100 if m50d else float("nan"),   # persen
    }


def mean_sd(vals):
    m = sum(vals) / len(vals)
    sd = statistics.stdev(vals) if len(vals) > 1 else float("nan")
    return m, sd


def fmt(x, nd):
    return "n/a" if x != x else f"{x:.{nd}f}"


per_fold, missing = {}, []
for fi in range(K):
    p = find_csv(fi)
    if p:
        per_fold[fi] = read_fold(p)
        print(f"[INFO] fold_{fi}: {p}")
    else:
        missing.append(fi)
if missing:
    print(f"[WARNING] CSV baseline fold {missing} tidak ditemukan — "
          f"rekap dihitung dari {len(per_fold)} fold yang tersedia.")
assert per_fold, "Tidak ada satu pun CSV baseline fold yang ditemukan. Cek Input notebook."

stats = {c: mean_sd([per_fold[fi][c] for fi in sorted(per_fold)]) for c in COLS}

with open(OUT_PATH, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["fold"] + COLS)
    for fi in sorted(per_fold):
        w.writerow([f"fold_{fi}"] + [f"{per_fold[fi][c]:.4f}" for c in COLS])
    w.writerow(["mean"] + [f"{stats[c][0]:.4f}" for c in COLS])
    w.writerow(["sd"] + [fmt(stats[c][1], 4) for c in COLS])
print(f"[INFO] Tersimpan: {OUT_PATH}\n")

print("| fold | " + " | ".join(HEADERS[c] for c in COLS) + " |")
print("|---|" + "---:|" * len(COLS))
for fi in sorted(per_fold):
    print(f"| fold_{fi} | " + " | ".join(fmt(per_fold[fi][c], ND[c]) for c in COLS) + " |")
print("| **mean ± SD** | " + " | ".join(
    f"**{fmt(stats[c][0], ND[c])} ± {fmt(stats[c][1], ND[c])}**" for c in COLS) + " |")


[INFO] fold_0: /kaggle/input/notebooks/keziameilanyt/base-kfold-0-eval/kd_output/student_baseline_overall_fold0_9class.csv
[INFO] fold_1: /kaggle/input/notebooks/keziameilany/base-kfold-1-eval/kd_output/student_baseline_overall_fold1_9class.csv
[INFO] fold_2: /kaggle/input/notebooks/stevieadrian/kfold2-eval/kd_output/student_baseline_overall_fold2_9class.csv
[INFO] fold_3: /kaggle/input/notebooks/keziameilanyt/base-kfold-3-eval/kd_output/student_baseline_overall_fold3_9class.csv
[INFO] fold_4: /kaggle/input/notebooks/keziameilany/base-kfold-4-eval/kd_output/student_baseline_overall_fold4_9class.csv
[INFO] Tersimpan: /kaggle/working/kfold_baseline_summary.csv

| fold | mAP@0.5 siang | mAP@0.5 malam | mAP@0.5:0.95 siang | mAP@0.5:0.95 malam | penurunan relatif (%) |
|---|---:|---:|---:|---:|---:|
| fold_0 | 0.5046 | 0.4731 | 0.2861 | 0.2531 | 6.24 |
| fold_1 | 0.5025 | 0.4792 | 0.2845 | 0.2523 | 4.64 |
| fold_2 | 0.4906 | 0.4586 | 0.2794 | 0.2470 | 6.52 |
| fold_3 | 0.5025 | 0.4849 | 0.2